In [181]:
import pandas as pd
import duckdb
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

In [182]:
PATH = "results"
PROC_DIR = Path(f"../artifacts/{PATH}")
#OUT_PATH = Path(f"output/{'_'.join(PATH.split('/'))}_info.tsv")
#OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(PROC_DIR)
files = sorted(PROC_DIR.glob("*_comparison.csv"))
files

../artifacts/results


[PosixPath('../artifacts/results/level_01_daily_total_sales_model_comparison.csv'),
 PosixPath('../artifacts/results/level_04_daily_dept_sales_model_comparison.csv'),
 PosixPath('../artifacts/results/level_06_daily_store_sales_model_comparison.csv'),
 PosixPath('../artifacts/results/level_09_daily_store_dept_sales_model_comparison.csv')]

In [183]:
sample = files[0]
sample

PosixPath('../artifacts/results/level_01_daily_total_sales_model_comparison.csv')

In [190]:
df_results = []
for file in files:
    filename = file.stem
    parts = filename.split("__")
    base = parts[0]
    split = base.split("_")
    stats = {"level": split[1], "grainly": split[2]}
    df = pd.read_csv(file)
    df['level'] = stats['level']
    df['level'] = df['level'].astype(int)
    df['grainly'] = stats['grainly']

    df_results.append(df)

df_results = pd.concat(df_results, ignore_index=True)

df_results['score'] = df_results['wrmsse'] + 0.01 * df_results['fit_time']

df_results.sort_values('score')

,model,category,wape,wrmsse,mae,rmse,smape,bias,rmsle,tracking_signal,spec,mase,fit_time,level,grainly,score
0,XGBoost,ML,0.038860,0.366459,1642.696847,2166.756650,0.039003,-0.000388,0.049984,0.279383,1744.734410,0.386270,2.931755,1,daily,0.395776
3,Seasonal naive (28d),Naive,0.057615,0.520748,2435.535714,3079.021650,0.058385,-0.006386,0.074777,3.103629,3488.616071,0.572701,0.006029,1,daily,0.520808
5,Theta,Statistical,0.058848,0.543849,2487.624439,3215.609114,0.058448,-0.006916,0.073627,3.290505,6332.190327,0.584949,0.138919,1,daily,0.545238
2,ETS (Holt-Winters),Statistical,0.057486,0.506892,2430.068402,2997.095171,0.057519,-0.014464,0.069258,7.045020,10098.978409,0.571415,5.068504,1,daily,0.557577
1,SARIMA,Statistical,0.056911,0.501102,2405.771822,2962.861762,0.056779,-0.016928,0.067699,8.328342,11536.470336,0.565702,6.928417,1,daily,0.570386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,LightGBM,ML,0.433533,2.129352,261.806505,464.539965,0.510944,-0.164729,0.934481,744.740452,20877.100011,2.616431,8.943759,9,daily,2.218790
77,CatBoost,ML,0.267558,1.626413,161.575519,300.090614,0.335103,-0.158422,0.425828,1160.523595,7308.338518,1.887386,92.913472,9,daily,2.555548
83,Ridge,ML,0.482790,2.548786,291.552735,519.977012,0.547839,-0.287942,0.910185,1168.968326,14622.064038,3.147331,5.743673,9,daily,2.606222
82,HistGradientBoosting,ML,0.512661,2.519306,309.591446,465.272971,0.606533,-0.064492,1.045624,246.563583,21420.041477,3.170071,11.218402,9,daily,2.631490


In [ ]:
df_mean_level = df_results.groupby(['level']).mean(numeric_only=True).reset_index().round(3)

df_mean_level['models'] =  df_results.groupby(['level'])['model'].nunique().values

df_mean_level

In [ ]:
sns.barplot(df_mean_level, x='level', y='wape')
plt.show()

In [ ]:
sns.barplot(df_mean_level, x='level', y='fit_time')
plt.show()

In [ ]:
sns.barplot(df_mean_level, x='level', y='wrmsse')
plt.axhline(1, color='red', linestyle='--')
plt.show()

# Levels

In [ ]:
levels = df_results['level'].unique()
df_results = df_results.sort_values(['level','category','model'])

def plot_metric_level(metric, df_results=df_results, levels=levels):
    
    for level in levels:
        df_level = df_results.query(f'level == {level}')[['level','category','model', metric]]
        df_level = df_level.sort_values(metric)
        palette = {'Naive': '#55A868', 'ML': '#4C72B0', 'Statistical': '#DD8452'}
        orden = ['ML', 'Naive', 'Statistical']
        
        plt.figure(figsize=(14, 5))
        plt.title(f"nivel={level} metrica={metric}")
        
        sns.barplot(df_level, x='model', y=metric, hue='category', hue_order=orden, palette=palette)
        
        plt.xticks(rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()

In [ ]:
df_results.columns

In [ ]:
plot_metric_level('wrmsse')

In [ ]:
plot_metric_level('wape')

In [ ]:
plot_metric_level('fit_time')